In [ ]:
from environment import Environment
from nnsight import LanguageModel
import sys
import torch as t
from getpass import getpass
import os

# from keys import REPLICATE
# REPLICATE_API_TOKEN = getpass()
# os.environ["REPLICATE_API_TOKEN"] = "REPLICATE"

# !git clone https://github.com/jbloomAus/mats_sae_training.git
# %cd mats_sae_training
# !pip install -r requirements.txt

sys.path.append("../mats_sae_training")

from sae_training.sparse_autoencoder import SparseAutoencoder

t.set_grad_enabled(False)

In [ ]:
model = LanguageModel("openai-community/gpt2", device_map='auto', dispatch=True)

In [ ]:
from huggingface_hub import hf_hub_download

REPO_ID = "jbloom/GPT2-Small-SAEs"

sae_list = []
n_layers = 12

local_dir = "../jbloom_dictionaries"

for layer in range(n_layers):
    filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"
    resid_sae = hf_hub_download(repo_id = REPO_ID, filename = filename, local_dir=local_dir)

    save_path = f"{local_dir}/{filename}"
    sae = SparseAutoencoder.load_from_pretrained(save_path)
    sae.to("cuda:0")
    
    sae_list.append(sae)

print("Loaded dictionaries")

In [ ]:
from config import EnvConfig, ExplainerConfig, CondenserConfig, DetectionScorerConfig, GenerationScorerConfig

env_config = EnvConfig()
explainer_cfg = ExplainerConfig()
condenser_cfg = CondenserConfig()
d_scorer_cfg = DetectionScorerConfig()
gen_scorer_cfg = GenerationScorerConfig()

env = Environment(
    model=model, 
    sae_list=sae_list,
    cfg=env_config,
    provider="openai"
)

In [ ]:
import threading

def feature_thread(environment, layer, feature_id, explainer_cfg, condenser_cfg, d_scorer_cfg, gen_scorer_cfg):
    environment.execute(layer, feature_id, explainer_cfg, condenser_cfg, d_scorer_cfg, gen_scorer_cfg)

# Assuming initialization of Environment, config, and other necessary components here
env = Environment(model, sae_list, env_config, "openai")
env.load_cache(10)

# Example of running features in separate threads
threads = []

layer = 10
for feature_id in [7000]:  # Assuming you want to process 10 features
    thread = threading.Thread(target=feature_thread, args=(env, layer, feature_id, explainer_cfg, condenser_cfg, d_scorer_cfg, gen_scorer_cfg))
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()